In [4]:
import os
import re
from pathlib import Path
from collections import Counter
from multiprocessing import Pool, cpu_count

import orjson  # High-speed JSON library
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# Compressed author and hotel url into integer IDs and mapping dictionery
# compressed into parquet, 
# deleted title column - we have text column anyways

# Filter reviews missing rating, text, or property_dict
# Count the number of reviews per author and hotel and filter those with less than 3 reviews
# process feature dictionery into separate columns

# also removed text

In [5]:
# === 1. CONFIGURATION ===
# --- Paths ---
INPUT_PATH = Path("HotelRec.txt")
CORE_PARQUET_PATH = Path("HotelRec_core_fast.parquet")
TEXT_PARQUET_PATH = Path("HotelRec_text_fast.parquet")
HOTEL_MAP_PATH = Path("hotel_id_mapping.json")
AUTHOR_MAP_PATH = Path("author_id_mapping.json")

# --- Parameters ---
MIN_REVIEWS = 3
CHUNK_SIZE = 200_000  # Number of lines each worker process gets at a time
NUM_PROCESSES = cpu_count()  # Use all available CPU cores

# --- Fixed Schema ---
# Use a predefined, normalized list of properties. As requested, we only use these.
FIXED_PROPERTIES_NORMALIZED = sorted([
    "cleanliness", "location", "rooms", "service", "sleep_quality", "value"
])
# Create a reverse map to find the original keys during processing
# This handles variations like "sleep quality" -> "sleep_quality"
PROPERTY_NORM_MAP = {
    re.sub(r"[^\w]+", "_", k.lower()).strip("_"): k 
    for k in ["cleanliness", "location", "rooms", "service", "sleep quality", "value"]
}

In [6]:
def process_chunk_for_counts(chunk_of_lines):
    """Parses a chunk of lines and returns counts of hotels and authors."""
    hotel_counter = Counter()
    author_counter = Counter()
    for line in chunk_of_lines:
        try:
            # Use orjson.loads for speed
            r = orjson.loads(line)
            h, a = r.get("hotel_url"), r.get("author")
            if h and a:
                hotel_counter[h] += 1
                author_counter[a] += 1
        except (orjson.JSONDecodeError, AttributeError):
            continue
    return hotel_counter, author_counter

In [7]:
def process_chunk_for_writing(args):
    """
    Processes a chunk of lines to generate final records for Parquet files.
    Filters, maps IDs, and structures the data.
    """
    chunk_of_lines, review_id_start, valid_hotels, valid_authors, hotel_id_map, author_id_map = args
    
    core_buffer = []
    text_buffer = []
    
    for i, line in enumerate(chunk_of_lines):
        try:
            r = orjson.loads(line)
            h, a = r.get("hotel_url"), r.get("author")

            # Fast filtering using set lookups
            if not (r.get("text") and h in valid_hotels and a in valid_authors):
                continue
            
            review_id = review_id_start + i
            pdict = r.get("property_dict") or {}
            
            # Build core record
            core_rec = {
                "review_id": review_id,
                "hotel_id": hotel_id_map[h],
                "author_id": author_id_map[a],
                "date": r.get("date"), # Keep as string, as requested
                "rating": r.get("rating"),
            }
            # Add fixed properties, defaulting to None if missing
            for norm_key in FIXED_PROPERTIES_NORMALIZED:
                original_key = PROPERTY_NORM_MAP.get(norm_key)
                core_rec[norm_key] = pdict.get(original_key)
            
            core_buffer.append(core_rec)
            text_buffer.append({"review_id": review_id, "text": r["text"]})
            
        except (orjson.JSONDecodeError, AttributeError):
            continue
            
    return core_buffer, text_buffer

In [8]:
# === 3. MAIN EXECUTION LOGIC ===

def main():
    """Main function to run the parallel preprocessing."""
    print(f"Starting analysis with {NUM_PROCESSES} processes.")

    # --- PHASE 1: Parallel Counting (Map-Reduce) ---
    print("\n--- Phase 1: Counting reviews in parallel ---")
    
    # Read the file and create an iterator of chunks
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        line_chunks = iter(lambda: [f.readline() for _ in range(CHUNK_SIZE)], [''])

        total_hotel_counter = Counter()
        total_author_counter = Counter()

        # Create a process pool
        with Pool(processes=NUM_PROCESSES) as pool:
            # Map chunks to worker processes and reduce the results
            map_results = pool.imap_unordered(process_chunk_for_counts, line_chunks)
            
            # Use tqdm for a progress bar
            for hotel_counter, author_counter in tqdm(map_results, desc="Counting chunks"):
                total_hotel_counter.update(hotel_counter)
                total_author_counter.update(author_counter)

    # Filter to get valid entities
    valid_hotels = {h for h, c in total_hotel_counter.items() if c >= MIN_REVIEWS}
    valid_authors = {a for a, c in total_author_counter.items() if c >= MIN_REVIEWS}
    print(f"✅ Phase 1 complete. Kept {len(valid_hotels):,} hotels and {len(valid_authors):,} authors.")

    # Create final ID mappings
    hotel_id_map = {hotel: i for i, hotel in enumerate(valid_hotels)}
    author_id_map = {author: i for i, author in enumerate(valid_authors)}

    # --- PHASE 2: Parallel Processing and Writing ---
    print("\n--- Phase 2: Processing and writing to Parquet in parallel ---")
    
    # Define schemas once
    core_fields = [
        pa.field("review_id", pa.int64()), pa.field("hotel_id", pa.int32()),
        pa.field("author_id", pa.int32()), pa.field("date", pa.string()),
        pa.field("rating", pa.float32())
    ]
    for key in FIXED_PROPERTIES_NORMALIZED:
        core_fields.append(pa.field(key, pa.float32()))
    core_schema = pa.schema(core_fields)
    text_schema = pa.schema([pa.field("review_id", pa.int64()), pa.field("text", pa.string())])

    core_writer = pq.ParquetWriter(CORE_PARQUET_PATH, core_schema, compression='snappy')
    text_writer = pq.ParquetWriter(TEXT_PARQUET_PATH, text_schema, compression='zstd') # zstd is great for text

    review_id_counter = 0
    with open(INPUT_PATH, "r", encoding="utf-8") as f, Pool(processes=NUM_PROCESSES) as pool:
        # We need to package arguments for the worker function
        def arg_generator():
            nonlocal review_id_counter
            for chunk in iter(lambda: [f.readline() for _ in range(CHUNK_SIZE)], ['']):
                args = (chunk, review_id_counter, valid_hotels, valid_authors, hotel_id_map, author_id_map)
                review_id_counter += len(chunk)
                yield args
        
        # Process chunks and write to Parquet
        map_results = pool.imap_unordered(process_chunk_for_writing, arg_generator())

        for core_data, text_data in tqdm(map_results, desc="Processing chunks"):
            if core_data:
                core_table = pa.Table.from_pylist(core_data, schema=core_schema)
                core_writer.write_table(core_table)
                
                text_table = pa.Table.from_pylist(text_data, schema=text_schema)
                text_writer.write_table(text_table)

    core_writer.close()
    text_writer.close()
    print(f"✅ Phase 2 complete. Data written to Parquet files.")
    
    # --- Save mappings ---
    with open(HOTEL_MAP_PATH, "w") as f_h, open(AUTHOR_MAP_PATH, "w") as f_a:
        orjson.dump(hotel_id_map, f_h)
        orjson.dump(author_id_map, f_a)
    print("✅ ID mappings saved.")

In [2]:
# === Paths ===
input_path = Path("HotelRec.txt")
# NEW: Define separate output files for core data and text data
core_parquet_path = Path("HotelRec_core.parquet")
text_parquet_path = Path("HotelRec_text.parquet")
hotel_map_path = Path("hotel_id_mapping.json")
author_map_path = Path("author_id_mapping.json")

# === Phase 1: Count hotels/authors ===
# This phase is efficient and remains largely the same.
hotel_counter = Counter()
author_counter = Counter()

print("Starting Phase 1: Counting entities...")
with open(input_path, "r", encoding="utf-8") as fin:
    for line in fin:
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        h = r.get("hotel_url")
        a = r.get("author")
        if h and a:
            hotel_counter[h] += 1
            author_counter[a] += 1

# --- Filter entities with at least 3 reviews ---
valid_hotels = {h for h, c in hotel_counter.items() if c >= 3}
valid_authors = {a for a, c in author_counter.items() if c >= 3}

print(f"✅ Phase 1 complete.")
print(f"Total unique hotels: {len(hotel_counter):,}, authors: {len(author_counter):,}")
print(f"Kept {len(valid_hotels):,} hotels and {len(valid_authors):,} authors with >= 3 reviews.\n")

Starting Phase 1: Counting entities...
✅ Phase 1 complete.
Total unique hotels: 365,057, authors: 21,891,404
Kept 365,007 hotels and 4,081,761 authors with >= 3 reviews.



In [3]:
# === Phase 2: Define schemas and stream-process data ===

# --- NEW: Define a fixed, normalized set of properties ---
FIXED_PROPERTIES = [
    "cleanliness", "location", "rooms", "service", "sleep quality", "value"
]

def normalize_key(k):
    return re.sub(r"[^\w]+", "_", k.lower()).strip("_")

# Create a normalized map for our fixed properties
prop_norm_map = {prop: normalize_key(prop) for prop in FIXED_PROPERTIES}
norm_props_ordered = [prop_norm_map[k] for k in FIXED_PROPERTIES]

# --- Pre-build ID maps for efficiency ---
hotel_id_map = {hotel: i for i, hotel in enumerate(valid_hotels)}
author_id_map = {author: i for i, author in enumerate(valid_authors)}

# --- Define explicit, efficient schemas for our TWO output files ---

# Schema for the slim, core data file
core_fields = [
    pa.field("review_id", pa.int64(), nullable=False),
    pa.field("hotel_id", pa.int32(), nullable=False),
    pa.field("author_id", pa.int32(), nullable=False),
    pa.field("timestamp", pa.int64()), # For Unix timestamp
    pa.field("rating", pa.float32()),
]
for norm_key in norm_props_ordered:
    core_fields.append(pa.field(norm_key, pa.float32()))
core_schema = pa.schema(core_fields)

# Schema for the text data file
text_schema = pa.schema([
    pa.field("review_id", pa.int64(), nullable=False),
    pa.field("text", pa.string())
])

# --- Stream data, process, and write to Parquet files ---
chunk_size = 1_000_000
core_buffer, text_buffer = [], []
core_writer, text_writer = None, None
review_id_counter = 0

print("Starting Phase 2: Processing reviews and writing to slim Parquet files...")
with open(input_path, "r", encoding="utf-8") as fin:
    for line in fin:
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue

        # Filter out records
        h, a = r.get("hotel_url"), r.get("author")
        if not all([r.get("text"), r.get("rating") is not None, h in valid_hotels, a in valid_authors]):
            continue

        pdict = r.get("property_dict") or {}

        # Core record processing
        core_rec = {
            "review_id": review_id_counter,
            "hotel_id": hotel_id_map[h],
            "author_id": author_id_map[a],
            # Convert date to timestamp; handle errors gracefully
            "timestamp": int(pd.to_datetime(r.get("date"), errors='coerce').timestamp()) if r.get("date") else None,
            "rating": r["rating"]
        }
        for raw_k, norm_k in prop_norm_map.items():
            core_rec[norm_k] = pdict.get(raw_k)
        
        core_buffer.append(core_rec)

        # Text record processing
        text_buffer.append({"review_id": review_id_counter, "text": r["text"]})
        review_id_counter += 1

        # When buffer is full, write it to Parquet
        if len(core_buffer) >= chunk_size:
            # Write core data
            df_core = pd.DataFrame(core_buffer)
            table_core = pa.Table.from_pandas(df_core, schema=core_schema, preserve_index=False)
            if core_writer is None:
                core_writer = pq.ParquetWriter(core_parquet_path, schema=core_schema, compression="snappy")
            core_writer.write_table(table_core)
            core_buffer.clear()
            
            # Write text data
            df_text = pd.DataFrame(text_buffer)
            table_text = pa.Table.from_pandas(df_text, schema=text_schema, preserve_index=False)
            if text_writer is None:
                text_writer = pq.ParquetWriter(text_parquet_path, schema=text_schema, compression="zstd")
            text_writer.write_table(table_text)
            text_buffer.clear()


# Final flush for any remaining records
if core_buffer:
    # Write core data
    df_core = pd.DataFrame(core_buffer)
    table_core = pa.Table.from_pandas(df_core, schema=core_schema, preserve_index=False)
    if core_writer is None:
        core_writer = pq.ParquetWriter(core_parquet_path, schema=core_schema, compression="snappy")
    core_writer.write_table(table_core)
    
    # Write text data
    df_text = pd.DataFrame(text_buffer)
    table_text = pa.Table.from_pandas(df_text, schema=text_schema, preserve_index=False)
    if text_writer is None:
        text_writer = pq.ParquetWriter(text_parquet_path, schema=text_schema, compression="zstd")
    text_writer.write_table(table_text)

# Close writers
if core_writer: core_writer.close()
if text_writer: text_writer.close()

print(f"✅ Phase 2 complete. Core data saved to '{core_parquet_path}' and text to '{text_parquet_path}'.")

# --- Save mapping dictionaries ---
with open(hotel_map_path, "w") as f:
    json.dump(hotel_id_map, f, indent=2)
with open(author_map_path, "w") as f:
    json.dump(author_id_map, f, indent=2)

print(f"✅ ID mappings saved.")

Starting Phase 2: Processing reviews and writing to slim Parquet files...


KeyboardInterrupt: 